# **Algorithmic Recourse: Практика**

**Привет, друзья!**

Через все уроки модуля «Deep Dive в Counterfactuals» шла Тамара: 63 года, бывшая бухгалтерка, подала заявку на позицию ML-инженера. Скрининговая модель приглашает на собеседование при $p_{\text{invite}} \ge 0.5$, а Тамаре дала $p = 0.03$. Признаки анкеты: возраст 63, пройдено курсов 2, проектов в портфолио 3, коммерческого опыта 0.5 года. Возраст изменить нельзя, и модель на него почти не смотрит; курсы, проекты и опыт Тамара может нарастить.

В уроках каждый метод отвечал Тамаре по отдельности. Здесь мы соберем ответы в одном месте и сравним их по свойствам из урока «Метрики и главный эмпирический факт»: validity, proximity, sparsity, plausibility, actionability, robustness.

В этой практике вы:
- соберете данные и модель-лес из уроков и проверите, что у Тамары $p = 0.03$;
- найдете минимальное изменение по Wachter перебором по сетке значений и увидите, как вес $\lambda$ решает, станет ли контрфактуал валидным;
- получите с DiCE три разных плана с замороженным возрастом и посчитаете близость, разреженность и разнообразие;
- добавите притяжение к прототипу класса «приглашена» и посмотрите, как контрфактуал становится типичнее и дальше от Тамары;
- построите FACE своими руками (k-NN-граф, плотность KDE, Дейкстра) и сравните путь шагами с прыжком Wachter;
- зададите стоимость действий в духе Ustun, найдете самый дешевый план и случай, когда recourse нет;
- проверите каждый найденный план на 15 переобученных моделях и на шуме исполнения, как в ROAR и PRR;
- сведете все в одну таблицу свойств.

С DiCE на другой задаче (регрессия цен домов) работает тетрадь `PRACTICE_counterfactual_explanations`; эта тетрадь — про свойства recourse на одном человеке.

**Приятного кодинга!**

### Какие библиотеки мы используем и почему не другие

В уроках модуля названы четыре табличные библиотеки. На Colab (Python 3.13, numpy 2.1.3) из них работает одна, поэтому остальные методы мы пишем коротко сами — по формулам из уроков.

| Библиотека | Что в ней есть | На Colab |
|---|---|---|
| `dice-ml` 0.12 | DiCE: genetic, random, kdtree | ставится, используем |
| `alibi` 0.9.6 | Wachter, `CounterfactualProto` | установка откатывает numpy 2.1.3 → 1.26.4, Pillow → 10.4.0, transformers 5.18.0 → 4.57.6 и blis → 0.7.11 и ломает остальное окружение; Wachter и прототипы пишем сами перебором по сетке значений |
| CARLA (`carla-recourse` 0.0.5) | Growing Spheres, REVISE, C-CHVAE, FACE, ROAR | не ставится: требует tensorflow 1.14, для которой нет сборок под Python 3.13 (и numpy 1.19, torch 1.7); FACE и проверку устойчивости пишем сами |
| OmniXAI 1.3.1 | обертка над DiCE и другими объяснениями | установка откатывает numpy до 1.23.5 и scikit-learn до 1.1.3 |
| `actionable-recourse` 1.0.1 | Ustun: целочисленное программирование | ставится вместе с `cplex` и `mip==1.12`, но падает при первом вызове `Flipset`: обращается к `np.float_`, которого нет в numpy 2; к тому же метод рассчитан на линейную модель, а у Тамары лес. Стоимость действий считаем полным перебором целочисленных действий |

Все это проверено установкой в окружение с версиями пакетов Colab на 10.10.2026.

In [ ]:
!pip install dice-ml==0.12 -q

In [ ]:
import random, time, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from scipy.stats import gaussian_kde
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
import dice_ml

pd.set_option('display.precision', 3)

## 1. Данные и модель из уроков

Код повторяет шаг «Growing Spheres для Тамары» урока 2: те же 2500 синтетических анкет с сидом 0, тот же скоринг (возраст в нем не участвует), та же модель — стандартизация и случайный лес из 200 деревьев. Мы только завернули генерацию в функцию `make_data(seed)`: в разделе 7 она понадобится для переобучения модели на новых данных, как функция `make(seed)` из шага «Робастный recourse для Тамары» урока 5. При `seed=0` получаются ровно те же данные, что в уроках.

In [ ]:
cont = ["age", "courses", "projects", "exp"]
skill = ["courses", "projects", "exp"]           # что Тамара может менять

def make_data(seed, n=2500):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "age":      rng.normal(42, 12, n).clip(22, 70),
        "courses":  rng.normal(4, 2, n).clip(0, 15),
        "projects": rng.normal(3, 2, n).clip(0, 20),
        "exp":      rng.normal(3, 2.5, n).clip(0, 20)})
    score = 0.5*df.courses + 0.7*df.projects + 0.6*df.exp + rng.normal(0, 1, n)  # возраст НЕ участвует
    df["invited"] = (score > np.median(score)).astype(int)
    return df

def make_model(df):
    return Pipeline([
        ("sc", ColumnTransformer([("s", StandardScaler(), cont)])),
        ("clf", RandomForestClassifier(n_estimators=200, random_state=0))
    ]).fit(df[cont], df["invited"])

df = make_data(0)
model = make_model(df)

tamara = np.array([63., 2., 3., 0.5])              # возраст, курсы, проекты, опыт
def p(x, m=model): return m.predict_proba(pd.DataFrame([x], columns=cont))[0, 1]
def p_many(X, m=model): return m.predict_proba(pd.DataFrame(X, columns=cont))[:, 1]

print("p(Тамара приглашена) =", round(p(tamara), 3))
print("доля приглашенных в данных:", df.invited.mean())
df.head()

Проверим обещание «модель на возраст не смотрит». В скоринге возраста нет, но лес все равно может делать по нему разбиения на шуме.

In [ ]:
imp = model.named_steps["clf"].feature_importances_
print("важность признаков в лесу:", dict(zip(cont, imp.round(3).tolist())))
print("p при другом возрасте и тех же навыках:",
      {a: round(float(p([a, 2., 3., 0.5])), 3) for a in (25, 40, 63, 70)})

Лес отдал возрасту около 9% важности, но $p$ Тамары почти не зависит от возраста: от 0.025 до 0.05 на всем диапазоне. Значит, ответ «помолодейте» Тамаре не поможет, даже если бы был возможен, — и возраст во всех методах ниже заморожен на 63.

### Как мы будем мерить свойства

Каждый найденный контрфактуал $x'$ мы опишем одними и теми же числами (их определения — в уроке «Метрики и главный эмпирический факт»):

- **validity** — $p(x') \ge 0.5$ у текущей модели;
- **proximity** — расстояние $L_1$ по навыкам, каждый навык поделен на свой MAD (медианное абсолютное отклонение), как у Wachter; меньше — ближе к Тамаре;
- **sparsity** — сколько из трех навыков изменилось;
- **plausibility** — среднее расстояние до 5 ближайших реальных анкет в стандартизованных навыках; меньше — типичнее (у самой Тамары это 0.177);
- **actionability** — возраст не тронут и ни один навык не уменьшился; для таких планов считаем стоимость в месяцах усилий (раздел 6);
- **robustness** — доля переобученных моделей и доля зашумленных исполнений, при которых план остается валидным (раздел 7).

In [ ]:
mad = (df[skill] - df[skill].median()).abs().median().values
sd = df[skill].std().values
knn = NearestNeighbors(n_neighbors=5).fit(df[skill].values / sd)
unit_cost = np.array([1., 2., 12.])             # месяцев на +1 курс, +1 проект, +1 год опыта (раздел 6)

def proximity(x):    return np.abs((x[1:] - tamara[1:]) / mad).sum()
def sparsity(x):     return int((np.abs(x[1:] - tamara[1:]) > 1e-9).sum())
def plausibility(x): return knn.kneighbors([x[1:] / sd])[0].mean()
def actionable(x):   return bool(x[0] == tamara[0] and (x[1:] >= tamara[1:] - 1e-9).all())
def cost(x):         return float((x[1:] - tamara[1:]) @ unit_cost)

def describe(x):
    return {"courses": round(x[1], 2), "projects": round(x[2], 2), "exp": round(x[3], 2),
            "p": round(p(x), 3), "valid": p(x) >= 0.5, "proximity": round(proximity(x), 2),
            "sparsity": sparsity(x), "plausibility": round(plausibility(x), 3),
            "actionable": actionable(x)}

found = {}                                       # сюда складываем контрфактуалы всех методов
pd.DataFrame([describe(tamara)], index=["Тамара"])

## 2. Wachter: минимальное изменение

Базовая постановка из шага «Что мы уже знаем о контрфактуалах?» урока 1:

$$\arg\min_{x'}\ \max_{\lambda}\ \lambda\big(f(x') - y'\big)^2 + d(x, x'),$$

где $d$ — наша proximity ($L_1$ с масштабом по MAD), $y' = 1$. В Alibi эту задачу решает градиентный спуск (для черного ящика градиент оценивается численно), и тот же поиск лежит в основе `CounterfactualProto`. У леса градиента нет, а навыков всего три, поэтому мы заменим спуск полным перебором по сетке значений с шагом 0.1: курсы от 0 до 8, проекты от 0 до 9, опыт от 0 до 8 лет, возраст 63. Это около 600 тысяч точек, лес считает их за несколько секунд.

In [ ]:
t0 = time.time()
c, pr, e = np.meshgrid(np.arange(0, 8.01, 0.1), np.arange(0, 9.01, 0.1), np.arange(0, 8.01, 0.1), indexing="ij")
G = np.column_stack([np.full(c.size, 63.), c.ravel(), pr.ravel(), e.ravel()])
PG = p_many(G)
DG = (np.abs(G[:, 1:] - tamara[1:]) / mad).sum(1)       # proximity каждой точки сетки
print(len(G), "точек, %.1f с" % (time.time() - t0))

Сначала посмотрим, что делает вес $\lambda$: при фиксированном $\lambda$ минимизируем $\lambda(p(x') - 1)^2 + d(x, x')$ по всей сетке.

In [ ]:
rows = []
for lam in [1, 10, 100, 1000]:
    k = np.argmin(lam * (PG - 1)**2 + DG)
    rows.append({"lambda": lam, **describe(G[k])})
pd.DataFrame(rows).set_index("lambda")

При $\lambda = 1$ штраф за прогноз дешевле любого шага, и оптимум — сама Тамара с $p = 0.03$: контрфактуал невалиден. С ростом $\lambda$ точка уходит за границу, а при больших $\lambda$ — глубже в область приглашения, дальше от Тамары. Поэтому в формуле стоит $\max_\lambda$: вес растят, пока прогноз не станет нужным. В пределе это задача «самая близкая точка среди валидных».

**Задание 1.** Найдите среди точек сетки с $p \ge 0.5$ точку с минимальной proximity `DG` и сохраните ее в `x_wachter`.

*Вопрос для самопроверки:* какие навыки меняет контрфактуал Wachter, какое у него $p$ и сколько навыков он трогает? Сравните с Growing Spheres из шага «Growing Spheres для Тамары» урока 2 (курсы $2\to2.2$, проекты $3\to4.4$, опыт $0.5\to1.6$).

<details><summary>Проверить себя</summary>

Контрфактуал Wachter: курсы 2 (без изменений), проекты $3 \to 4.4$, опыт $0.5 \to 1.6$ года, $p = 0.505$, proximity 1.69; меняются 2 навыка из 3. Growing Spheres в уроке 2 пришел почти в ту же точку (проекты 4.4, опыт 1.6), но тронул и курсы ($2 \to 2.2$): там точку искали случайным сэмплированием в шаре, а полный перебор с расстоянием $L_1$ оставляет курсы на месте. Решение: `x_wachter = G[np.argmin(np.where(PG >= 0.5, DG, np.inf))]`.
</details>

In [ ]:
x_wachter = ...  # Ваш код здесь
found["Wachter"] = x_wachter
pd.DataFrame([describe(x_wachter)], index=["Wachter"])

Контрфактуал Wachter стоит на самой границе: $p$ едва выше 0.5. Это свойство минимального решения, и в разделе 7 мы увидим, во что оно обходится.

## 3. DiCE: несколько разных планов

Сначала повторим шаг «DiCE для Тамары» урока 2 как есть: генетический поиск, три контрфактуала, возраст исключен из `features_to_vary`, запрос записан целыми числами. В уроке поиск шел без сида, и третий контрфактуал менялся от запуска к запуску; здесь мы фиксируем `random` и `numpy`, и все три плана воспроизводятся. Рядом с колонкой `invited`, которую печатает DiCE, поставим $p$ от самой модели.

In [ ]:
d = dice_ml.Data(dataframe=df, continuous_features=cont, outcome_name="invited")
m = dice_ml.Model(model=model, backend="sklearn")
query_int = pd.DataFrame([{"age": 63, "courses": 2, "projects": 3, "exp": 0.5}])   # как в уроке

random.seed(0); np.random.seed(0)
cf_int = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query_int, total_CFs=3, desired_class=1, features_to_vary=skill)
out = cf_int.cf_examples_list[0].final_cfs_df.copy()
print(out.dtypes.to_dict())
out["p модели"] = p_many(out[cont].values.astype(float)).round(3)
out

Первые два плана совпадают с таблицей урока: курсы 3, проекты 3, опыт 3.2 и курсы 2, проекты 4, опыт 1.6. DiCE пишет у них `invited = 1`, а модель дает $p = 0.285$ и $p = 0.255$: оба плана **невалидны**. Причина в типах: в запросе курсы и проекты записаны целыми числами, и DiCE приводит найденные контрфактуалы к типам запроса, отбрасывая дробную часть уже после поиска. Внутри поиска курсов было, например, 3.4, а в выдаче осталось 3, и точка вернулась за границу. Отсюда правило: валидность контрфактуала проверяют прогнозом модели, а не колонкой, которую печатает библиотека.

Запишем запрос дробными числами и повторим поиск.

In [ ]:
query = pd.DataFrame([{"age": 63., "courses": 2., "projects": 3., "exp": 0.5}])

random.seed(0); np.random.seed(0)
t0 = time.time()
cf_gen = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill)
print("genetic: %.2f с" % (time.time() - t0))
dice_gen = cf_gen.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
pd.DataFrame([describe(x) for x in dice_gen], index=["DiCE-1", "DiCE-2", "DiCE-3"])

Теперь все три плана валидны. Первые два — те же пути, что в уроке, но без отброшенной дробной части: DiCE-1 — курсы 3.4 и опыт 3.2 (в уроке курсов осталось 3), DiCE-2 — курсы 2.5, проекты 4.8 и опыт 1.6 (в уроке курсов 2 и проектов 4). Третий план — 5.5 курса при 2.9 проекта: DiCE уменьшил число проектов. `features_to_vary` заморозил возраст, но направление изменений не ограничивает; такой план формально валиден, а выполнить его нельзя (проект из портфолио не «отменяют»). Для этого в DiCE есть `permitted_range`.

Для сравнения запустим тот же запрос методом `random`: он сэмплирует точки в допустимой области и оставляет валидные.

In [ ]:
random.seed(0); np.random.seed(0)
cf_rnd = dice_ml.Dice(d, m, method="random").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill, random_seed=0)
dice_rnd = cf_rnd.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
pd.DataFrame([describe(x) for x in dice_rnd], index=["random-1", "random-2", "random-3"])

**Задание 2.** Разнообразие набора DiCE меряет определителем матрицы схожести (шаг «Формальная постановка DiCE» урока 2). Посчитаем более простую величину: среднее попарное расстояние $L_1$ с масштабом по MAD между планами набора. Допишите функцию `diversity(X)` для массива планов `X` (строки — планы, столбцы — `cont`).

*Вопрос для самопроверки:* у какого метода, `genetic` или `random`, выше средняя proximity планов и выше diversity набора? Какие числа получились?

<details><summary>Проверить себя</summary>

`genetic`: средняя proximity 2.55, diversity 3.42; `random`: средняя proximity 8.19, diversity 10.82. Метод `random` дальше от Тамары и разнообразнее: он сэмплирует всю допустимую область и не притягивает планы к объекту. Решение: среднее по парам $i < j$ от `np.abs((X[i, 1:] - X[j, 1:]) / mad).sum()`.
</details>

In [ ]:
def diversity(X):
    # Ваш код здесь
    ...

for name, X in [("genetic", dice_gen), ("random", dice_rnd)]:
    print("%-8s средняя proximity = %.2f   средняя sparsity = %.2f   diversity = %.2f"
          % (name, np.mean([proximity(x) for x in X]), np.mean([sparsity(x) for x in X]), diversity(X)))

Теперь запретим уменьшать навыки: `permitted_range` задает коридор, который начинается с текущего значения Тамары.

In [ ]:
random.seed(0); np.random.seed(0)
cf_pr = dice_ml.Dice(d, m, method="genetic").generate_counterfactuals(
    query, total_CFs=3, desired_class=1, features_to_vary=skill,
    permitted_range={"courses": [2, 15], "projects": [3, 20], "exp": [0.5, 20]})
dice_pr = cf_pr.cf_examples_list[0].final_cfs_df[cont].values.astype(float)
print("diversity = %.2f" % diversity(dice_pr))
pd.DataFrame([describe(x) for x in dice_pr], index=["DiCE+range-1", "DiCE+range-2", "DiCE+range-3"])

С коридором все три плана выполнимы, но меняют только опыт, и diversity набора упала с 3.42 до 0.47. Ограничение по actionability купили ценой diversity: это первый конфликт свойств в тетради.

In [ ]:
found["DiCE-1"], found["DiCE-2"], found["DiCE-3"] = dice_gen
found["DiCE+range"] = dice_pr[0]

## 4. Прототипы: контрфактуал рядом с типичными приглашенными

Прототип строим как в шаге «Прототип и латент для Тамары» урока 3: среднее 25 ближайших к Тамаре реальных приглашенных анкет (расстояние в стандартизованных признаках), возраст оставляем 63.

In [ ]:
sd_all = df[cont].std().values
inv = df[df.invited == 1][cont].values
proto = inv[np.argsort(np.linalg.norm((inv - tamara) / sd_all, axis=1))[:25]].mean(0)
proto[0] = 63                                   # возраст неизменяем — держим
pd.DataFrame([describe(proto)], index=["прототип"])

Числа совпадают с уроком (курсы 3.3, проекты 3.8, опыт 2.2), и стоит заметить одно: сам прототип модель **не** приглашает, $p = 0.215$. Ближайшие к Тамаре приглашенные получили приглашение отчасти благодаря шуму в скоринге, и их среднее лежит еще до границы. Значит, прототип годится как направление, а не как ответ: его используют как штраф внутри поиска, а валидность обеспечивает условие $p \ge 0.5$.

Возьмем цель прототипного метода из шага «Что добавляется к цели» урока 3 без автоэнкодера (его роль частично играет сам прототип):

$$L(x') = d(x, x') + \theta\,\Big\lVert \frac{x' - \text{proto}}{\text{sd}} \Big\rVert_2^2 \quad\text{при}\quad p(x') \ge 0.5.$$

**Задание 3.** Для каждого $\theta$ из списка найдите на сетке из раздела 2 валидную точку с минимальной $L$. Квадрат стандартизованного расстояния до прототипа для всех точек сетки уже посчитан в `DP`.

*Вопрос для самопроверки:* как меняются proximity и plausibility контрфактуала при переходе от $\theta = 0$ к $\theta = 10$?

<details><summary>Проверить себя</summary>

При $\theta = 0$ (и при $\theta = 1$) ответ совпадает с Wachter: proximity 1.69, plausibility 0.228. При $\theta = 10$: курсы 3.5, проекты 4.2, опыт 2.5, proximity 3.16 (почти вдвое больше), plausibility 0.125 (меньше, то есть типичнее; у самой Тамары 0.177). Решение: `k = np.argmin(np.where(PG >= 0.5, DG + theta * DP, np.inf))`.
</details>

In [ ]:
DP = (((G[:, 1:] - proto[1:]) / sd)**2).sum(1)
rows, proto_cf = [], {}
for theta in [0, 1, 3, 10]:
    k = ...  # Ваш код здесь: индекс точки сетки
    proto_cf[theta] = G[k]
    rows.append({"theta": theta, **describe(G[k]), "до прототипа": round(np.sqrt(DP[k]), 2)})
pd.DataFrame(rows).set_index("theta")

При малых $\theta$ штраф прототипа слабее, чем шаг по сетке, и ответ совпадает с Wachter. При $\theta = 10$ контрфактуал уходит к прототипу и оказывается в более плотной области анкет, но proximity выросла с 1.69 до 3.16, а plausibility-дистанция упала с 0.228 до 0.125. Это компромисс урока 3: правдоподобность покупается близостью.

In [ ]:
found["Прототип θ=10"] = proto_cf[10]

## 5. FACE руками: путь по плотным областям

CARLA с FACE на Colab не ставится, поэтому повторяем код шага «Три требования к recourse Тамары» урока 4: вершины графа — Тамара и 2500 реальных анкет (только навыки), ребра — 15 ближайших соседей в стандартизованных навыках, вес ребра $-\log \hat p\big(\tfrac{x_i + x_j}{2}\big)\cdot \lVert x_i - x_j \rVert$ с плотностью KDE в середине ребра. Цель — анкета, которую модель при возрасте Тамары приглашает с $p \ge 0.6$ и которая сама лежит в плотной половине данных. Путь ищет Дейкстра.

Единственное отличие от урока: плотность в серединах ребер мы считаем одним вызовом KDE для всех ребер сразу, а не в цикле. Числа от этого не меняются, а ячейка работает в несколько раз быстрее.

In [ ]:
t0 = time.time()
X = df[skill].values.astype(float)
proba = model.predict_proba(df[cont].assign(age=63.))[:, 1]   # прогноз для навыков анкеты при возрасте Тамары

pts = np.vstack([tamara[1:], X]); N = len(pts)  # Тамара — узел 0
Z = (pts - X.mean(0)) / X.std(0)                # стандартизуем для расстояний
kde = gaussian_kde(X.T)                         # плотность данных

dist, idx = NearestNeighbors(n_neighbors=16).fit(Z).kneighbors(Z)
rows = np.repeat(np.arange(N), 15); cols = idx[:, 1:].ravel(); dd = dist[:, 1:].ravel()
dens = kde(((pts[rows] + pts[cols]) / 2).T)
w = np.maximum(-np.log(dens + 1e-12) * dd, 1e-6)  # -log(плотность) · расстояние
Gr = csr_matrix((w, (rows, cols)), shape=(N, N)); Gr = Gr.maximum(Gr.T)

dens_all = kde(X.T)
targets = np.where((proba >= 0.6) & (dens_all >= np.quantile(dens_all, 0.5)))[0] + 1
dpath, pred = dijkstra(Gr, indices=0, return_predecessors=True)
best = targets[np.argmin(dpath[targets])]

path = [best]
while path[-1] != 0: path.append(pred[path[-1]])
path = path[::-1]
print("%.1f с" % (time.time() - t0))
for k in path: print(np.round(pts[k], 2))
x_face = np.r_[63., pts[best]]
found["FACE"] = x_face
pd.DataFrame([describe(x_face)], index=["FACE"])

Путь совпадает с уроком: Тамара $[2, 3, 0.5] \to [2, 3.55, 0.88] \to [2.18, 4, 1.51] \to [2.39, 4.37, 1.8]$, у конечной анкеты $p = 0.625$ (лес в `describe` дает то же число, ведь конечная точка — навыки реальной анкеты при возрасте 63). Каждая промежуточная точка — реальный человек из выборки.

Сравним, по каким местам проходят путь FACE и прыжок Wachter. Для каждого отрезка возьмем 50 точек и посмотрим самую низкую плотность KDE на нем; чтобы число читалось, переведем его в долю реальных анкет, у которых плотность еще ниже.

In [ ]:
def min_density_share(points):
    ts = np.linspace(0, 1, 50)[:, None]
    low = min(kde((a + ts * (b - a)).T).min() for a, b in zip(points[:-1], points[1:]))
    return (dens_all < low).mean()

face_pts = pts[path]
jump_pts = np.vstack([tamara[1:], x_wachter[1:]])
for name, P in [("FACE", face_pts), ("Wachter", jump_pts)]:
    print("%-8s длина в навыках = %.2f, до конца по прямой = %.2f, "
          "доля анкет с плотностью ниже самой разреженной точки пути = %.3f"
          % (name, sum(np.linalg.norm(b - a) for a, b in zip(P[:-1], P[1:])),
             np.linalg.norm(P[-1] - P[0]), min_density_share(P)))

In [ ]:
plt.figure(figsize=(7, 5))
sc = plt.scatter(X[:, 1], X[:, 2], c=proba, cmap="Blues", s=6, vmin=0, vmax=1)
plt.colorbar(sc, label="p(приглашена) при возрасте 63")
plt.plot(face_pts[:, 1], face_pts[:, 2], "-o", color="#eb6834", lw=2, ms=8, label="путь FACE")
plt.plot(jump_pts[:, 1], jump_pts[:, 2], "--s", color="#1baf7a", lw=2, ms=8, label="прыжок Wachter")
plt.scatter([tamara[2]], [tamara[3]], color="black", s=60, zorder=5, label="Тамара")
plt.xlim(1.5, 6.5); plt.ylim(0, 4)
plt.xlabel("проекты"); plt.ylabel("опыт, лет")
plt.title("Анкеты, путь FACE и прыжок Wachter (проекция на проекты и опыт)")
plt.legend(loc="upper right"); plt.show()

На рисунке точки — реальные анкеты в проекции на проекты и опыт, цвет — вероятность приглашения при возрасте 63. Оранжевая ломаная — путь FACE по реальным анкетам, зеленая пунктирная — прыжок Wachter напрямую к границе. Здесь оба маршрута лежат в населенной области: самая разреженная точка пути FACE плотнее, чем у 66% реальных анкет, прыжка Wachter — чем у 55%. Тамара стоит в густой середине данных, и пропасти между ней и приглашенными нет; путь FACE идет по чуть более плотным местам. За реальную конечную точку и реальные промежуточные шаги FACE платит близостью: до его конечной точки 1.93 по прямой в навыках против 1.78 у Wachter, proximity 2.07 против 1.69. Если бы плотной тропы не было, Дейкстра не нашла бы пути к целевым анкетам — это и есть ответ FACE «recourse нет».

Еще одна деталь конечной точки FACE. Цель выбиралась по условию $p \ge 0.6$, но это $p$ дает лес, который видел эту анкету при обучении. Обучим модель заново на тех же данных без одной этой анкеты и спросим ее снова.

In [ ]:
model_wo = make_model(df.drop(index=best - 1))     # узел best — строка best - 1 в df
print("p модели из уроков = %.3f | p модели без этой анкеты = %.3f"
      % (p(x_face), p(x_face, model_wo)))

Без своей анкеты лес дает конечной точке FACE $p = 0.35$, ниже порога. Случайный лес на обучающих объектах завышает уверенность: каждая анкета попала в бутстреп-выборки большинства деревьев. Поэтому порог уверенности $t_p$ в FACE стоит проверять прогнозами, полученными без этого объекта (out-of-fold), — иначе «реальный приглашенный человек» окажется приглашенным только для модели, которая его запомнила. В разделе 7 это проявится еще раз.

## 6. Стоимость действий в духе Ustun

Расстояние в признаках — геометрия, а стоимость — усилие Тамары (шаг «Actionable Recourse» урока 4). Зададим грубые цены в месяцах: курс — 1 месяц, проект — 2 месяца, год опыта — 12 месяцев (его надо прожить). Множество допустимых действий $A(x)$ — целые числа: от 0 до 6 новых курсов, от 0 до 6 новых проектов и от 0 до 10 полугодий опыта; уменьшать навыки и трогать возраст нельзя. Это 539 действий, и полный перебор по ним заменяет целочисленное программирование из пакета `actionable-recourse`: на конечном множестве он находит точный минимум и так же точно доказывает, что recourse нет.

**Задание 4.** Для каждого действия из `acts` посчитайте его стоимость в месяцах (`act_cost`) и найдите самое дешевое валидное действие.

*Вопрос для самопроверки:* какой план самый дешевый, сколько он стоит месяцев и какое у него $p$? Сколько месяцев стоил бы план Wachter по тем же ценам?

<details><summary>Проверить себя</summary>

Самый дешевый план — четыре новых курса (курсов $2 \to 6$, проекты и опыт без изменений): 4 месяца, $p = 0.59$. План Wachter по тем же ценам стоит 16 месяцев (1.4 проекта по 2 месяца и 1.1 года опыта по 12 месяцев). Решение: `act_cost = acts @ np.array([1, 2, 6])` (полугодие опыта стоит 6 месяцев), `k = np.argmin(np.where(PA >= 0.5, act_cost, np.inf))`.
</details>

In [ ]:
acts = np.array(list(itertools.product(range(7), range(7), range(11))))   # +курсы, +проекты, +полугодия опыта
XA = np.tile(tamara, (len(acts), 1))
XA[:, 1:] += acts * np.array([1., 1., 0.5])
PA = p_many(XA)
act_cost = ...  # Ваш код здесь: стоимость каждого действия в месяцах
k = ...         # Ваш код здесь: индекс самого дешевого валидного действия
x_ustun = XA[k]
found["Ustun"] = x_ustun
print("самый дешевый план:", acts[k], "→", x_ustun, "| стоимость =", act_cost[k],
      "мес. | p =", round(PA[k], 3))
print("стоимость плана Wachter по тем же ценам: %.1f мес." % cost(x_wachter))

order = np.argsort(np.where(PA >= 0.5, act_cost, np.inf))[:6]
pd.DataFrame({"+courses": acts[order, 0], "+projects": acts[order, 1], "+exp": acts[order, 2] * 0.5,
              "месяцев": act_cost[order], "p": PA[order].round(3)})

Самый дешевый план не похож на ближайший. Wachter двигал проекты и опыт: к ним модель чувствительнее (в скоринге у них веса 0.7 и 0.6 против 0.5 у курсов), и на единицу расстояния они дают больший прирост $p$. По усилию же дешевле всего курсы. Proximity и actionability — разные величины, и выбор между ними делает цена, а не геометрия.

Теперь сузим $A(x)$ до реальных ограничений ближайшего полугодия: Тамара успеет не больше двух курсов и двух проектов, а опыт без работы не растет.

In [ ]:
allowed = (acts[:, 0] <= 2) & (acts[:, 1] <= 2) & (acts[:, 2] == 0)
print("допустимых действий:", allowed.sum(), "| из них валидных:", (PA[allowed] >= 0.5).sum(),
      "| максимальное p =", round(PA[allowed].max(), 3))

Ни одно из 9 допустимых действий не дает $p \ge 0.5$: при таких ограничениях recourse для Тамары не существует, и перебор это доказывает, а не предполагает. Так выглядит ответ, ради которого Ustun и соавторы и формулировали задачу через множество действий: «за полгода ничего из доступного не поможет, нужен опыт».

## 7. Робастность в духе ROAR и PRR

Recourse — обещание во времени (урок 5). Проверим каждый план с двух сторон:

- **сдвиг модели (ROAR):** компания переобучает скрининг на новых данных. Как функция `make(seed)` из шага «Робастный recourse для Тамары», генерируем данные с сидами 1–15 и обучаем 15 моделей; доля моделей, у которых $p \ge 0.5$, — robustness к сдвигу;
- **шум исполнения (PRR):** Тамара выполнит план неточно. Добавляем к навыкам гауссов шум со стандартными отклонениями 0.5 курса, 0.5 проекта и 0.7 года опыта (как в уроке), 500 раз; доля валидных исполнений — robustness к шуму.

Рядом поставим запас (margin) $p - 0.5$ и proximity.

In [ ]:
t0 = time.time()
models = [make_model(make_data(s)) for s in range(1, 16)]
print("15 моделей за %.0f с" % (time.time() - t0))

**Задание 5.** Допишите `share_models(x)` — долю из 15 переобученных моделей `models`, у которых $p(x) \ge 0.5$. Функция шума `share_noise(x)` уже написана: она каждый раз берет один и тот же сид, поэтому все планы проверяются на одинаковых возмущениях.

*Вопрос для самопроверки:* какая доля переобученных моделей приглашает Тамару по плану Wachter и по рецепту урока 5 «с буфером» (курсы 3, проекты 4, опыт 6.5)? Чему равны их margin?

<details><summary>Проверить себя</summary>

План Wachter приглашают 0 из 15 переобученных моделей (доля 0.0), его margin 0.005. План «с буфером» приглашают 15 из 15 (доля 1.0), его margin 0.475. Решение: `return np.mean([p(x, mm) >= 0.5 for mm in models])`.
</details>

In [ ]:
def share_models(x):
    # Ваш код здесь
    ...

def share_noise(x, n=500):
    g = np.random.default_rng(3)                # шум исполнения по навыкам
    V = np.array(x, float) + g.normal(0, [0, 0.5, 0.5, 0.7], size=(n, 4))
    V[:, 1:] = np.clip(V[:, 1:], 0, None)
    return (p_many(V) >= 0.5).mean()

found["урок 5: у границы"] = np.array([63., 3., 4., 3.5])
found["урок 5: с буфером"] = np.array([63., 3., 4., 6.5])

true_thr = np.median(0.5*df.courses + 0.7*df.projects + 0.6*df.exp)   # порог скоринга без шума
def score_margin(x): return 0.5*x[1] + 0.7*x[2] + 0.6*x[3] - true_thr

rob = pd.DataFrame({name: {"p": round(p(x), 3), "margin": round(p(x) - 0.5, 3), "proximity": round(proximity(x), 2),
                           "запас по скорингу": round(score_margin(x), 2),
                           "доля моделей": round(share_models(x), 3), "доля при шуме": round(share_noise(x), 3)}
                    for name, x in found.items()}).T
rob

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, col, title in [(ax[0], "доля моделей", "сдвиг модели (15 переобученных)"),
                      (ax[1], "доля при шуме", "шум исполнения (500 исполнений)")]:
    a.scatter(rob["margin"], rob[col], s=60, color="#2a78d6")
    for name, r in rob.iterrows():
        a.annotate(name, (r["margin"], r[col]), fontsize=8, xytext=(4, 3), textcoords="offset points")
    a.set_xlabel("margin = p − 0.5 у текущей модели"); a.set_ylabel("доля валидных")
    a.set_title(title); a.set_ylim(-0.05, 1.08); a.set_xlim(-0.03, 0.62)
plt.tight_layout(); plt.show()

На двух графиках каждая точка — план одного метода: по горизонтали его запас над порогом у текущей модели, по вертикали доля переобученных моделей (слева) и зашумленных исполнений (справа), при которых план остается валидным. План Wachter с margin 0.005 не выживает ни на одной из 15 переобученных моделей и только в 19% зашумленных исполнений; план «с буфером» с margin 0.475 выживает везде, но его proximity 4.99 против 1.69 у Wachter.

Связь margin с робастностью при этом грубая: у FACE margin 0.125, а переобученные модели не приглашают его ни разу (конечная точка взята из обучающей выборки, раздел 5), у прототипа при $\theta = 10$ margin 0, а доля моделей 0.533. Данные у нас синтетические, и мы знаем правило, по которому их размечали, поэтому в таблице есть столбец «запас по скорингу» — $0.5\cdot\text{курсы} + 0.7\cdot\text{проекты} + 0.6\cdot\text{опыт}$ минус медиана этой суммы по данным. Доли переобученных моделей идут почти строго по нему: планы с отрицательным запасом по скорингу держатся на шумовом «кармане» текущего леса и при переобучении пропадают. В реальной задаче этого правила нет, поэтому ROAR и PRR проверяют сами возмущения, а не margin одной модели.

Последний шаг — встроить требование робастности в сам поиск. На целочисленных действиях из раздела 6 найдем самый дешевый план, валидный **на всех** 15 переобученных моделях и на текущей (ROAR: худшая модель из множества), и самый дешевый план, который выживает хотя бы в 95% зашумленных исполнений (PRR с $\delta = 0.05$).

In [ ]:
t0 = time.time()
worst = np.min([p_many(XA, mm) for mm in models + [model]], axis=0)     # худшая модель для каждого действия
k_roar = np.argmin(np.where(worst >= 0.5, act_cost, np.inf))
cheap = np.argsort(np.where(PA >= 0.5, act_cost, np.inf))
k_prr = next(k for k in cheap if share_noise(XA[k]) >= 0.95)
found["ROAR (худшая из 16)"] = XA[k_roar]
found["PRR (δ=0.05)"] = XA[k_prr]
for name, k in [("Ustun", np.argmin(np.where(PA >= 0.5, act_cost, np.inf))), ("ROAR", k_roar), ("PRR", k_prr)]:
    print("%-6s +%s → %s | стоимость = %d мес. | p = %.3f" % (name, acts[k], XA[k], act_cost[k], PA[k]))
print("%.1f с" % (time.time() - t0))

## 8. Итоговая таблица свойств

Соберем все найденные планы в одну таблицу. Стрелка в заголовке говорит, куда лучше: ↑ — больше, ↓ — меньше. Стоимость в месяцах посчитана только для выполнимых планов (возраст не тронут, ни один навык не уменьшился).

In [ ]:
summary = pd.DataFrame({name: {
    "courses": round(x[1], 2), "projects": round(x[2], 2), "exp": round(x[3], 2),
    "validity ↑": p(x) >= 0.5,
    "proximity ↓": round(proximity(x), 2),
    "sparsity ↓": sparsity(x),
    "plausibility ↓": round(plausibility(x), 3),
    "actionable ↑": actionable(x),
    "месяцев ↓": round(cost(x), 1) if actionable(x) else np.nan,
    "доля моделей ↑": round(share_models(x), 3),
    "доля при шуме ↑": round(share_noise(x), 3)} for name, x in found.items()}).T
summary

Таблица подтверждает главный вывод урока «Метрики и главный эмпирический факт»: ни один метод не лучший по всем столбцам сразу.
- Самая малая proximity у Wachter (1.69), но его не приглашает ни одна из 15 переобученных моделей, а при шуме исполнения он валиден в 18.6% случаев.
- Самая высокая plausibility (дистанция 0.125 и 0.129) у прототипа при $\theta = 10$ и у FACE; FACE при этом не приглашает ни одна переобученная модель, а прототипный план стоит 27.9 месяца.
- Меньше всего навыков меняют DiCE с коридором и Ustun (по одному), но первый стоит 40.8 месяца, а второй, самый дешевый (4 месяца), держится только на 1 из 15 переобученных моделей.
- DiCE-3 валиден, но невыполним: в нем уменьшается число проектов.
- Полную робастность к сдвигу модели дают план «с буфером» (75 месяцев) и ROAR (8 месяцев); у ROAR при этом доля валидных исполнений при шуме 0.668, а PRR (тоже 8 месяцев), наоборот, валиден в 99% исполнений и на 11 из 15 моделей.

Выбор для Тамары зависит от того, что ей дороже: быстрее всего — четыре курса по плану Ustun, с риском, что после переобучения скрининга их не хватит; надежнее всего — план ROAR, два курса и три проекта за 8 месяцев, который выдерживает все 16 моделей.

## **Выводы**

В этой практике мы ответили Тамаре всеми методами модуля на одних данных и одной модели и увидели, что:
- минимальный по расстоянию контрфактуал (Wachter) стоит на границе решения и хрупок при переобучении модели и неточном исполнении;
- валидность надо проверять прогнозом модели: DiCE привел планы к целым типам запроса и напечатал `invited = 1` у невалидных точек;
- DiCE дает меню планов, но `features_to_vary` не запрещает уменьшать навыки; коридор `permitted_range` делает планы выполнимыми и одновременно однообразными;
- прототип задает направление к типичным приглашенным, но сам может оказаться невалидным; притяжение к нему повышает правдоподобность ценой близости;
- FACE ведет по реальным анкетам и показывает путь шагами, а платит за это близостью; порог уверенности для его цели стоит проверять прогнозами модели, обученной без этой анкеты;
- стоимость действий меняет ответ сильнее, чем метрика расстояния, и полный перебор по множеству действий умеет доказать, что recourse нет;
- запас над порогом покупает робастность к сдвигу модели и шуму исполнения, и платит за нее близостью и месяцами;
- ни один метод не лучший по всем свойствам сразу: выбор зависит от того, ошибиться в каком свойстве дороже в вашей задаче.